# Bronze layer: raw TPC-H ingestion

The destination is **workspace.tpch_bronze**. It is created on the first run.
All eight source tables are copied without casts, renames, filters, or deduplication.
Existing target tables are overwritten, never appended to. Delta writes are atomic per table, not for all eight tables together.


## 1. Configure the destination and import repository code


In [0]:
import importlib
import os
import sys
from pathlib import Path

os.environ["DB_CATALOG"] = "workspace"
os.environ["SCHEMA_PREFIX"] = "tpch"

notebook_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)
workspace_path = (
    notebook_path
    if notebook_path.startswith("/Workspace/")
    else "/Workspace" + notebook_path
)
repo_root = Path(workspace_path).parent.parent
if not (repo_root / "config" / "env.py").is_file():
    raise RuntimeError(
        f"Open this notebook inside its Git folder. Repository not found: {repo_root}"
    )
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import env as project_env
from src.bronze import ingest

importlib.reload(project_env)
importlib.reload(ingest)
if Path(project_env.__file__).resolve() != (repo_root / "config" / "env.py").resolve():
    raise RuntimeError(
        "A different 'config' module is loaded; restart the Python session."
    )
if project_env.CATALOG != "workspace" or project_env.BRONZE_SCHEMA != "tpch_bronze":
    raise RuntimeError("Unexpected Bronze destination; check the config overrides.")
print(f"Repository: {repo_root}")
print(f"Source: {project_env.SOURCE_CATALOG}.{project_env.SOURCE_SCHEMA}")
print(f"Destination: {project_env.CATALOG}.{project_env.BRONZE_SCHEMA}")

## 2. Confirm source access (read-only)


In [0]:
spark.sql("SHOW TABLES IN samples.tpch").show(truncate=False)
spark.table("samples.tpch.region").show(truncate=False)

## 3. Ingest and verify all eight tables

This cell performs the writes. Count, column order/name/type, and Delta-format
checks run for every table. Exact value comparison for every large table is
available through `verify_contents=True`, but is disabled here to avoid costly
shuffles. These checks describe what they verify; matching counts alone do not
prove every value is identical.


In [0]:
results = ingest.ingest_raw_tpch(spark)
print(f"Ingestion completed: {len(results)}/8 tables verified.")

## 4. Demonstrate a rerun with exact content verification

Rewrite only `region`, then compare both directions with `EXCEPT ALL`.
This includes duplicate multiplicities and null values, unlike a distinct-only
comparison. The same overwrite logic is used for every table.


In [0]:
region_before = spark.table("workspace.tpch_bronze.region").count()
rerun_result = ingest.ingest_raw_tpch(
    spark,
    tables=("region",),
    verify_contents=True,
)[0]
if (
    rerun_result["bronze_rows"] != region_before
    or rerun_result["content_check"] != "PASS"
):
    raise RuntimeError(
        "Region rerun did not preserve the original row count and contents."
    )
print("RERUN_CHECK: PASS — region row count unchanged; exact contents match source.")

## 5. Show the final summary and copyable verification report


In [0]:
import json

summary_schema = """source_table STRING, target_table STRING,
source_rows BIGINT, bronze_rows BIGINT, row_count_match BOOLEAN,
schema_match BOOLEAN, format STRING, content_check STRING, status STRING"""
display(spark.createDataFrame(results, schema=summary_schema))
spark.sql("SHOW TABLES IN workspace.tpch_bronze").show(truncate=False)
if len(results) != 8 or any(row["status"] != "PASS" for row in results):
    raise RuntimeError("Not all eight Bronze tables passed verification.")
print("BRONZE_VERIFICATION_JSON")
print(
    json.dumps(
        {
            "destination": "workspace.tpch_bronze",
            "tables": results,
            "region_rerun": rerun_result,
            "region_rows_before_rerun": region_before,
        },
        indent=2,
    )
)
print("BRONZE_READY: 8/8 tables passed; region rerun passed.")